# MedLift-3D — Kaggle pipeline

End-to-end run on a Kaggle notebook. Built around three Kaggle facts:

* **12-hour session cap** — training uses `--max-hours 11`, checkpoints, and
  resumes automatically when you rerun the same cell in a new session.
* **No internet by default** — the projector is pure PyTorch, so nothing beyond
  Kaggle's preinstalled stack is needed. Logging is CSV on disk, not wandb.
* **`/kaggle/working` is the only writable path** — auto-detected.

Enable **GPU (T4 x2 or P100)** in Settings → Accelerator before running.

> Order matters: run the gates first. Four failure modes in this problem are
> silent — they produce plausible loss curves and wrong reconstructions.

## 0 · Setup

Point `REPO` at the code. Either add this repository as a Kaggle *dataset* /
*GitHub* source, or clone it if internet is enabled.

In [ ]:
import os, sys, subprocess, pathlib

def find_input(slug, owner=None, base=pathlib.Path('/kaggle/input')):
    """Locate an attached input by slug.

    Kaggle mounts a dataset at /kaggle/input/<slug> on some accounts/dataset
    types and at /kaggle/input/datasets/<owner>/<slug> on others -- the layout
    is not consistent, so guessing one and hardcoding it breaks silently the
    moment it's wrong. This tries both, then falls back to a search, and on
    failure prints what IS actually attached rather than a bare path error.
    """
    candidates = [base / slug]
    if owner:
        candidates.append(base / 'datasets' / owner / slug)
    candidates += sorted(base.glob(f'datasets/*/{slug}'))
    candidates += sorted(base.glob(f'*/{slug}'))
    for c in candidates:
        if c.exists():
            return c
    have = sorted(str(p.relative_to(base)) for p in base.glob('*/*') if p.is_dir())
    have += sorted(str(p.relative_to(base)) for p in base.glob('*') if p.is_dir())
    raise FileNotFoundError(
        f"no input attached for slug={slug!r} owner={owner!r}. "
        f"Currently under /kaggle/input:\n  " + "\n  ".join(sorted(set(have)) or ["(nothing attached)"]))

try:
    REPO = find_input('medlift3d')                # attached as a Kaggle Dataset
except FileNotFoundError:
    REPO = pathlib.Path('/kaggle/working/medlift3d')
    if not REPO.exists():
        # Needs internet enabled; otherwise attach the repo as a dataset.
        subprocess.run(['git', 'clone', '--depth', '1',
                        'https://github.com/foyeznaeem/medlift3d.git', str(REPO)], check=True)

# Kaggle input is read-only, so work from a writable copy.
WORK = pathlib.Path('/kaggle/working')
if str(REPO).startswith('/kaggle/input'):
    subprocess.run(['cp', '-r', str(REPO), str(WORK / 'medlift3d')], check=True)
    REPO = WORK / 'medlift3d'

sys.path.insert(0, str(REPO / 'src'))
os.chdir(REPO)

import torch
print('repo   :', REPO)
print('torch  :', torch.__version__, '| cuda:', torch.cuda.is_available())
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        p = torch.cuda.get_device_properties(i)
        print(f'  gpu {i}: {p.name}  {p.total_memory/2**30:.1f} GiB')


## 1 · Gates — run these before spending any GPU time

| Gate | Asserts | Guards against |
|---|---|---|
| G1 | one canonical grid, affine never dropped | comparing volumes on different grids |
| G2 | `fp` differentiable, `bp` its exact adjoint | a projection loss with **no gradient** |
| G3 | water cylinder integral = `mu_water·2R` | HU/density confusion, scale errors |
| G4 | a *perfect* reconstruction scores perfectly | metrics that cannot detect anything |

If any gate fails, stop. Every one of these failures is silent and invalidates
every downstream number.

In [ ]:
!python scripts/run_gates.py

### 1b · Smoke test on synthetic phantoms — optional, but cheap

`make_phantoms.py` builds synthetic chest phantoms with nodules, so the whole
pipeline can be run end to end *before* committing to a multi-hour download.
Worth doing once: it exercises training, reconstruction, evaluation and all
three experiments on data that takes minutes to generate, and it sets `DATA`
so you can carry straight on through sections 3-7 on phantoms.

Skip it and go to section 2 if you already trust the pipeline and want real
data now. Section 2 re-points `DATA` at LIDC-IDRI either way.


In [ ]:
# 1b -- synthetic phantoms; skip if you are going straight to real data.
DATA = '/kaggle/working/data/phantom'

!python scripts/make_phantoms.py \
    --n-cases 60 \
    --shape 256 256 256 --spacing 1.5 1.5 1.5 \
    --views-a 16 32 --views-b 15 \
    --nodules 3 --device cuda \
    --out {DATA}


## 2 · Data — LIDC-IDRI

Fetched automatically from [TCIA](https://www.cancerimagingarchive.net/collection/lidc-idri/),
the collection's own host. Nothing to attach, no manifest, no API key.

**All the logic lives in `scripts/`, not in these cells.** That matters: cell 0
re-clones the repo, so a fix to the fetching code arrives just by re-running
this notebook. Cells that hold logic themselves go stale silently, because
Kaggle stores them separately from the repo.

Two points worth knowing:

* **pylidc already has the annotations.** It ships a 25 MB database holding all
  1,018 scans, 6,859 radiologist readings and 41,406 traced contours. Only the
  pixel data is missing, so only that is downloaded.
* **The directory layout is not negotiable.** pylidc looks for
  `<root>/<PatientID>/<StudyInstanceUID>/<SeriesInstanceUID>/*.dcm` and raises
  if `<root>/<PatientID>/` is absent. `fetch_lidc.py` writes exactly that.

Needs **Settings → Internet: On**. Start at `--batch 2` to prove the chain,
then raise it (~43 s per series).


In [ ]:
# 2-i -- fetch CT series from TCIA and write ~/.pylidcrc.
#
# pylidc has to be installed here rather than in a script, because the script
# that needs it cannot install it for itself. Everything else lives in
# scripts/, so it updates when cell 0 re-clones.
%pip install -q pylidc

# Raise --batch once this works; --skip N fetches a later slice of the collection.
!python scripts/fetch_lidc.py --batch 2 --out /kaggle/working/lidc_dicom


In [ ]:
# 2-ii -- ingest with pylidc: resample to the canonical grid, simulate the
# X-ray projections, write one .npz per case.
#
# Watch the last line: "ingested N cases (M with nodule masks)". M should be
# close to N. M == 0 means the masks are not coming through -- stop rather than
# training against no ground truth.
import pathlib, shutil

DATA = '/kaggle/working/data/lidc'
DICOM_ROOT = pathlib.Path('/kaggle/working/lidc_dicom')

!python scripts/prepare_lidc.py --source pylidc \
    --limit 2 --max-slice-thickness 1.5 \
    --out {DATA}

n_cases = len(list(pathlib.Path(DATA).glob('*.npz')))
print(f'\n{n_cases} case files in {DATA}')
if n_cases:
    shutil.rmtree(DICOM_ROOT, ignore_errors=True)   # the .npz is what we keep
    print(f'freed {DICOM_ROOT}')
else:
    print(f'KEEPING {DICOM_ROOT} -- ingest produced nothing, so the download '
          f'is preserved for a retry.')


## 3 · Train the prior

~48M params at 256², batch 8 with AMP ≈ 7 GB on a T4.

**Rerun this cell in a new session to resume** — it picks up from `last.pt`
with the optimiser, scaler, step and best-val intact. `--max-hours 11` stops
cleanly before Kaggle kills the session, so no progress is ever lost to the cap.

In [ ]:
!python scripts/train_prior.py \
    --data {DATA} \
    --out  /kaggle/working/runs/prior \
    --epochs 60 --batch-size 8 --base-dim 64 \
    --amp --workers 2 --max-hours 11

In [ ]:
import pandas as pd, matplotlib.pyplot as plt
log = pd.read_csv('/kaggle/working/runs/prior/log.csv')
ax = log.plot(x='epoch', y=['train_loss', 'val_loss'], figsize=(6, 3.4), grid=True)
ax.set_ylabel('eps-prediction MSE'); plt.tight_layout(); plt.show()
log.tail()

## 4 · Reconstruct

Baselines first — a learned prior has to beat them to justify its existence.
`--n-posterior 8` gives the mean reconstruction *and* the per-voxel uncertainty
map; it costs ~270 MB at 256³ and is the only thing that lets a reader tell
measured structure from structure the prior invented.

In [ ]:
# DATA was set in section 2 (LIDC-IDRI), or by the 1b smoke test
RECON = '/kaggle/working/runs/recon'
PRIOR = '/kaggle/working/runs/prior/best.pt'

for method in ['fbp', 'sirt_tv', 'cgls']:
    !python scripts/reconstruct.py --data {DATA} --out {RECON} \
        --track A16 --method {method} --n-iter 60 --device cuda

!python scripts/reconstruct.py --data {DATA} --out {RECON} \
    --track A16 --method diffusion --prior {PRIOR} \
    --n-steps 50 --n-posterior 8 --slice-batch 16 --device cuda

## 5 · Evaluate

Paired metrics, because this is a paired per-patient reconstruction task with
ground truth for every case — not FID/MMD, which measure distributional
similarity for *unconditional* generation. Nodule metrics are computed inside a
dilated bounding box, and PSNR/SSIM inside the lung mask.

In [ ]:
!python scripts/evaluate.py --recon {RECON} --data {DATA}

In [ ]:
from IPython.display import Image, display
import glob
for f in sorted(glob.glob(f'{RECON}/*/*.png'))[:6]:
    print(f.split('/')[-2], '·', f.split('/')[-1]); display(Image(f))

## 6 · The experiments that make the contribution

The architecture alone is not the novelty — R²-Gaussian, X-Gaussian, DOLCE and
DiffusionMBIR already occupy that ground. These three are:

1. **MDVC** — minimum detectable volume change vs view count and arc. Answers the
   Volume Doubling Time objective directly, and says where absolute volumetry
   stops being trustworthy.
2. **Hallucination audit** — insert / erase / present. Yields detection
   sensitivity and the **false-positive nodule rate**, which is the number a
   clinician cares about given the 96% LDCT false-positive rate.
3. **O5 ablation** — does the Gaussian parameterisation earn its place? A negative
   answer is a legitimate result.

> All three need the four-reader nodule masks, which is precisely why
> LIDC-IDRI is the dataset and not a centroid-only collection.


In [ ]:
!python scripts/mdvc.py --data {DATA} --out /kaggle/working/runs/mdvc \
    --tracks A32 A16 B15 --methods sirt_tv diffusion --prior {PRIOR} \
    --gains 0.05 0.10 0.20 0.30 0.50 --limit 5 --device cuda

In [ ]:
display(Image('/kaggle/working/runs/mdvc/mdvc.png'))

In [ ]:
!python scripts/hallucination.py --data {DATA} \
    --out /kaggle/working/runs/hallucination \
    --tracks A16 B15 --methods sirt_tv diffusion --prior {PRIOR} \
    --limit 5 --device cuda

In [ ]:
!python scripts/ablate_roi.py --data {DATA} \
    --out /kaggle/working/runs/ablate_roi \
    --track A16 --limit 3 --iters 800 --device cuda

## 7 · Collect outputs

Kaggle persists `/kaggle/working` (~20 GB). Checkpoints are large, so keep the
one you need and drop the rest before committing the notebook.

In [ ]:
import shutil, pathlib

for p in pathlib.Path('/kaggle/working/runs/prior').glob('last.pt'):
    print('consider deleting to save space:', p, f'{p.stat().st_size/2**20:.0f} MiB')

shutil.make_archive('/kaggle/working/medlift3d_results', 'zip',
                    '/kaggle/working/runs', )
print('bundled -> /kaggle/working/medlift3d_results.zip')